<a href="https://colab.research.google.com/github/nick1982ad/CS_Data_analysis/blob/main/Jupyter%20Notebooks/Scapy_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Монтрирование Google Drive (нужно для персистентного хранения данных - например, датасетов)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [122]:
!pip install scapy PyX

In [116]:
!apt update
!apt install tcpdump ghostscript texlive texlive-latex-extra

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,934 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
G

In [117]:
from scapy.all import *

# Основы работы с библиотекой Scapy

# Манипуляции с пакетами

- пакеты это объекты
- оператор `/` используется для инкапсуляции пакетов

---

In [5]:
packet = IP() / TCP()
Ether() / packet

<Ether  type=IPv4 |<IP  frag=0 proto=6 |<TCP  |>>>

In [6]:
packet

<IP  frag=0 proto=6 |<TCP  |>>

In [10]:
#getlayer позволяет вывести конкретный вложенный протокол
packet.getlayer(TCP)

<TCP  |>

- функция `ls()` показывает все поля пакета (протокола)



In [18]:
ls(Ether)

dst        : DestMACField                        = ('None')
src        : SourceMACField                      = ('None')
type       : XShortEnumField                     = ('36864')


In [17]:
ls(IP)

version    : BitField  (4 bits)                  = ('4')
ihl        : BitField  (4 bits)                  = ('None')
tos        : XByteField                          = ('0')
len        : ShortField                          = ('None')
id         : ShortField                          = ('1')
flags      : FlagsField                          = ('<Flag 0 ()>')
frag       : BitField  (13 bits)                 = ('0')
ttl        : ByteField                           = ('64')
proto      : ByteEnumField                       = ('0')
chksum     : XShortField                         = ('None')
src        : SourceIPField                       = ('None')
dst        : DestIPField                         = ('None')
options    : PacketListField                     = ('[]')


In [27]:
ls(TCP)

sport      : ShortEnumField                      = ('20')
dport      : ShortEnumField                      = ('80')
seq        : IntField                            = ('0')
ack        : IntField                            = ('0')
dataofs    : BitField  (4 bits)                  = ('None')
reserved   : BitField  (3 bits)                  = ('0')
flags      : FlagsField                          = ('<Flag 2 (S)>')
window     : ShortField                          = ('8192')
chksum     : XShortField                         = ('None')
urgptr     : ShortField                          = ('0')
options    : TCPOptionsField                     = ("b''")


- вывести иерархию протоколов для пакета

In [57]:
[l.__name__ for l in packet.layers()]

['IP', 'TCP']

- В scapy реализовано автоматическое разрешение DNS имен и MAC адресов

---

In [28]:
p = Ether() / IP(dst="www.secdev.org") / TCP(flags="F")
p.summary()

'Ether / IP / TCP 172.28.0.12:20 > Net("www.secdev.org/32"):80 F'

In [30]:
p.show()

###[ Ethernet ]###
  dst       = None
  src       = 02:42:ac:1c:00:0c
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = None
     tos       = 0x0
     len       = None
     id        = 1
     flags     = 
     frag      = 0
     ttl       = 64
     proto     = 6
     chksum    = None
     src       = 172.28.0.12
     dst       = Net("www.secdev.org/32")
     \options   \
###[ TCP ]###
        sport     = 20
        dport     = 80
        seq       = 0
        ack       = 0
        dataofs   = None
        reserved  = 0
        flags     = F
        window    = 8192
        chksum    = None
        urgptr    = 0
        options   = []



In [31]:
p.show2()

###[ Ethernet ]###
  dst       = 02:42:b5:04:b6:7b
  src       = 02:42:ac:1c:00:0c
  type      = IPv4
###[ IP ]###
     version   = 4
     ihl       = 5
     tos       = 0x0
     len       = 40
     id        = 1
     flags     = 
     frag      = 0
     ttl       = 64
     proto     = 6
     chksum    = 0x4388
     src       = 172.28.0.12
     dst       = 217.25.178.5
     \options   \
###[ TCP ]###
        sport     = 20
        dport     = 80
        seq       = 0
        ack       = 0
        dataofs   = 5
        reserved  = 0
        flags     = F
        window    = 8192
        chksum    = 0x5838
        urgptr    = 0
        options   = []



- доступ к различным полям пакета

---

In [32]:
print(p.dst)      # first layer with a dst field, i.e. Ether
print(p[IP].src)  # explicit access to the IP layer src field

# sprintf() supports Scapy own formats strings
print(p.sprintf("%Ether.src% > %Ether.dst%\n%IP.src% > %IP.dst%"))

None
172.28.0.12
02:42:ac:1c:00:0c > None
172.28.0.12 > Net("www.secdev.org/32")


принудительное вычисление значений всех полей пакета

In [37]:
raw_bytes = bytes(p)
computed = Ether(raw_bytes)
computed.dst

'02:42:b5:04:b6:7b'

- генерация массивов пакетов с разными значениями полей

---

In [46]:
p_list = [p for p in IP(src="10.0.0.1", ttl=(1,5)) / ICMP()]  # a sequence of values from 1 to 5
p_list

[<IP  frag=0 ttl=1 proto=1 src=10.0.0.1 |<ICMP  |>>,
 <IP  frag=0 ttl=2 proto=1 src=10.0.0.1 |<ICMP  |>>,
 <IP  frag=0 ttl=3 proto=1 src=10.0.0.1 |<ICMP  |>>,
 <IP  frag=0 ttl=4 proto=1 src=10.0.0.1 |<ICMP  |>>,
 <IP  frag=0 ttl=5 proto=1 src=10.0.0.1 |<ICMP  |>>]

In [41]:
[p for p in IP() / TCP(dport=[22, 80, 443])]  # specific values

[<IP  frag=0 proto=6 |<TCP  dport=22 |>>,
 <IP  frag=0 proto=6 |<TCP  dport=80 |>>,
 <IP  frag=0 proto=6 |<TCP  dport=443 |>>]

- проверка наличия заданного протокола в пакете

In [43]:
for p in p_list:
  if ICMP in p:
    print(p.ttl)

1
2
3
4
5


- преобразование массива пакетов в таблицу

In [47]:
import pandas as pd
packet_df = pd.DataFrame([p.fields for p in p_list if IP in p])
packet_df

,src,ttl,options
0,10.0.0.1,1,[]
1,10.0.0.1,2,[]
2,10.0.0.1,3,[]
3,10.0.0.1,4,[]
4,10.0.0.1,5,[]


# Живое взаимодействие с сетью

- функция `sr1()` позволяет отправить пакет и получить на него ответ
- Scapy автоматически сопоставляет запрос и ответ (если речь об известном протоколе)
    
---

In [63]:
p = sr1(IP(dst="8.8.8.8") / UDP() / DNS(rd=1, qd=DNSQR(qname="ya.ru", qtype="A")))
p[DNS].an


Received 1 packets, got 1 answers, remaining 0 packets


[<DNSRR  rrname=b'ya.ru.' type=A cacheflush=0 rclass=IN ttl=393 rdata=77.88.44.242 |>,
 <DNSRR  rrname=b'ya.ru.' type=A cacheflush=0 rclass=IN ttl=393 rdata=5.255.255.242 |>,
 <DNSRR  rrname=b'ya.ru.' type=A cacheflush=0 rclass=IN ttl=393 rdata=77.88.55.242 |>]

- Функция `sniff()` позволяет осуществлять захват трафика

---

In [64]:
help(sniff)

Help on function sniff in module scapy.sendrecv:

sniff(*args, **kwargs)
    Sniff packets and return a list of packets.

    Args:
        count: number of packets to capture. 0 means infinity.
        store: whether to store sniffed packets or discard them
        prn: function to apply to each packet. If something is returned, it
             is displayed.
             --Ex: prn = lambda x: x.summary()
        session: a session = a flow decoder used to handle stream of packets.
                 --Ex: session=TCPSession
                 See below for more details.
        filter: BPF filter to apply.
        lfilter: Python function applied to each packet to determine if
                 further action may be done.
                 --Ex: lfilter = lambda x: x.haslayer(Padding)
        offline: PCAP file (or list of PCAP files) to read packets from,
                 instead of sniffing them
        quiet:   when set to True, the process stderr is discarded
                 (default: 

In [65]:
s = sniff(count=2)
s

<Sniffed: TCP:2 UDP:0 ICMP:0 Other:0>

In [66]:
sniff(count=2, prn=lambda p: p.summary())

Ether / IP / TCP 172.28.0.12:8080 > 172.28.0.1:46734 PA / Raw
Ether / IP / TCP 172.28.0.1:46734 > 172.28.0.12:8080 A


<Sniffed: TCP:2 UDP:0 ICMP:0 Other:0>

- пакеты можно прочитать из pcap файла

---

In [95]:
pcap_p = rdpcap("/content/drive/MyDrive/DataAnalysis/http_espn_fail.pcapng")
len(pcap_p)

569

- метод `command()` выводит строку, которая позволяет создать точно такой же пакет

---

In [68]:
pcap_p[0].command()

"Ether(dst='c0:c1:c0:17:8c:e8', src='78:31:c1:cb:b2:56', type=2048)/IP(version=4, ihl=5, tos=0, len=58, id=9480, flags=0, frag=0, ttl=64, proto=17, chksum=37630, src='172.16.16.154', dst='4.2.2.1')/UDP(sport=57434, dport=53, len=38, chksum=31007)/DNS(id=45323, qr=0, opcode=0, aa=0, tc=0, rd=1, ra=0, z=0, ad=0, cd=0, rcode=0, qdcount=1, ancount=0, nscount=0, arcount=0, qd=[DNSQR(qname=b'www.espn.com.', qtype=1, unicastresponse=0, qclass=1)])"

- анализ протоколов в pcap файле

In [79]:
def get_proto_list(pcap_p):
  proto_list = set()
  for packet in pcap_p:
    proto_list = proto_list | set(l.__name__ for l  in packet.layers())
  return proto_list

In [80]:
get_proto_list(pcap_p)

{'DNS', 'Ether', 'IP', 'Padding', 'Raw', 'TCP', 'UDP'}

- использоване `sniff()` в offline режиме позволяет фильтровать уже имеющийся pcap файл

In [82]:
pcap_p = sniff(offline="/content/drive/MyDrive/DataAnalysis/http_espn_fail.pcapng", filter="port 53")
len(pcap_p)

14

In [83]:
get_proto_list(pcap_p)

{'DNS', 'Ether', 'IP', 'UDP'}

In [90]:
pcap_p[1][DNS].an

[<DNSRR  rrname=b'www.espn.com.' type=CNAME cacheflush=0 rclass=IN ttl=225 rdata=b'redir.espn.gns.go.com.' |>,
 <DNSRR  rrname=b'redir.espn.gns.go.com.' type=A cacheflush=0 rclass=IN ttl=223 rdata=68.71.212.158 |>]

- Функция `lsc()` выводит список доступных команд (функций)

---

```
>>> lsc()
IPID_count          : Identify IP id values classes in a list of packets
arpcachepoison      : Poison target's cache with (your MAC,victim's IP) couple
arping              : Send ARP who-has requests to determine which hosts are up
bind_layers         : Bind 2 layers on some specific fields' values
bridge_and_sniff    : Forward traffic between interfaces if1 and if2, sniff and return the
chexdump            :  Build a per byte hexadecimal representation
computeNIGroupAddr  : Compute the NI group Address. Can take a FQDN as input parameter
corrupt_bits        : Flip a given percentage or number of bits from a string
corrupt_bytes       : Corrupt a given percentage or number of bytes from a string
defrag              : defrag(plist) -> ([not fragmented], [defragmented],
defragment          : defrag(plist) -> plist defragmented as much as possible
dhcp_request        : --
[..]
```

In [ ]:
lsc()

In [96]:
from collections import defaultdict

# Итоговый словарь: {нормализованный 5-tuple: {"forward": N, "backward": N}}
biflows = defaultdict(lambda: {"forward": 0, "backward": 0})

for p in pcap_p:
    if IP not in p:
        continue

    proto = p[IP].proto

    if TCP in p:
        sport, dport = p[TCP].sport, p[TCP].dport
    elif UDP in p:
        sport, dport = p[UDP].sport, p[UDP].dport
    else:
        sport, dport = 0, 0

    src, dst = p[IP].src, p[IP].dst

    # Нормализация ключа — меньший IP всегда первый
    if (src, sport) < (dst, dport):
        key = (src, dst, proto, sport, dport)
        biflows[key]["forward"] += len(p[IP])
    else:
        key = (dst, src, proto, dport, sport)
        biflows[key]["backward"] += len(p[IP])


# Визуализация

- функция `bytes()` позволяет перевести пакет в его бинарное представление (как будет передан в сеть)

In [101]:
pkt = IP() / UDP() / DNS(qd=DNSQR())
repr(bytes(pkt))

"b'E\\x00\\x00=\\x00\\x01\\x00\\x00@\\x11|\\xad\\x7f\\x00\\x00\\x01\\x7f\\x00\\x00\\x01\\x005\\x005\\x00)\\xb6\\xd3\\x00\\x00\\x01\\x00\\x00\\x01\\x00\\x00\\x00\\x00\\x00\\x00\\x03www\\x07example\\x03com\\x00\\x00\\x01\\x00\\x01'"

Также scapy может вывести пакет в виде шестнадцатиричного дампа
   -  `hexdump` позволяет вывести содрежимое в human-readable варианте

In [102]:
hexdump(pkt)

0000  45 00 00 3D 00 01 00 00 40 11 7C AD 7F 00 00 01  E..=....@.|.....
0010  7F 00 00 01 00 35 00 35 00 29 B6 D3 00 00 01 00  .....5.5.)......
0020  00 01 00 00 00 00 00 00 03 77 77 77 07 65 78 61  .........www.exa
0030  6D 70 6C 65 03 63 6F 6D 00 00 01 00 01           mple.com.....


In [ ]:
pkt.show()

###[ IP ]### 
  version   = 4
  ihl       = None
  tos       = 0x0
  len       = None
  id        = 1
  flags     = 
  frag      = 0
  ttl       = 64
  proto     = udp
  chksum    = None
  src       = 127.0.0.1
  dst       = 127.0.0.1
  \options   \
###[ UDP ]### 
     sport     = domain
     dport     = domain
     len       = None
     chksum    = None
###[ DNS ]### 
        id        = 0
        qr        = 0
        opcode    = QUERY
        aa        = 0
        tc        = 0
        rd        = 1
        ra        = 0
        z         = 0
        ad        = 0
        cd        = 0
        rcode     = ok
        qdcount   = 1
        ancount   = 0
        nscount   = 0
        arcount   = 0
        \qd        \
         |###[ DNS Question Record ]### 
         |  qname     = 'www.example.com'
         |  qtype     = A
         |  qclass    = IN
        an        = None
        ns        = None
        ar        = None



  - графическое представление для изучения структуры пакета

In [124]:
pkt.canvas_dump()

ImportError: PyX and its dependencies must be installed

In [119]:
import pyx

In [120]:
pyx.__version__

'0.17'

In [ ]:
ans, unans = traceroute('www.secdev.org', maxttl=15)


Received 11 packets, got 11 answers, remaining 4 packets
   217.25.178.5:tcp80 
1  172.28.0.1      11 
2  192.178.254.91  11 
3  192.178.105.140 11 
7  198.32.118.57   11 
9  184.104.196.231 11 
10 184.104.189.18  11 
11 184.104.207.166 11 
12 217.25.179.111  11 
13 217.25.178.5    SA 
14 217.25.178.5    SA 
15 217.25.178.5    SA 


Different methods can be used to display the results, like `world_trace()` that uses the GeoIP module from [MaxMind](https://www.maxmind.com/)

---

## Scapy as a Python module

Scapy can be used to build your own tools like `ping6.py`:

---

In [ ]:
from scapy.all import *
import argparse

parser = argparse.ArgumentParser(description="A simple ping6")
parser.add_argument("ipv6_address", help="An IPv6 address")
args = parser.parse_args()

print sr1(IPv6(dst=args.ipv6_address) / ICMPv6EchoRequest(), verbose=0).summary()

```shell
sudo python ping6.py www.kame.net
IPv6 / ICMPv6 Echo Reply (id: 0x0 seq: 0x0)
```

## Answering Machines

Scapy can wait for a query, then send an answer with the `AnsweringMachine` object.

Two methods are mandatory:
1. `is_request()`: returns `True` if the packet is the expected query
2. `make_reply()`: returns the packet that will be sent by Scapy


Note: in the following example, the Wi-Fi interface must be put in *monitor* mode

In [ ]:
# Specify the Wi-Fi interface
conf.iface = "mon0"

# Create the Answering Machine
class ProbeRequest_am(AnsweringMachine):
  function_name = "pram"

  mac = "00:11:22:33:44:55"

  def is_request(self, pkt):
    return Dot11ProbeReq in pkt

  def make_reply(self, req):

    rep = RadioTap()
    # Note: depending on your Wi-Fi card, you might need something else than RadioTap()
    rep /= Dot11(addr1=req.addr2, addr2=self.mac, addr3=self.mac, ID=RandShort(), SC=RandShort())
    rep /= Dot11ProbeResp(cap="ESS", timestamp=time.time())
    rep /= Dot11Elt(ID="SSID",info="Scapy !")
    rep /= Dot11Elt(ID="Rates",info='\x82\x84\x0b\x16\x96')
    rep /= Dot11Elt(ID="DSset",info=chr(10))

    return rep

# Start the answering machine
#ProbeRequest_am()()  # uncomment to test

![](https://github.com/guedou/guedou.github.io/blob/master/talks/2022_GreHack/images/scapy_am.png?raw=1)

# X.509 Certificates Manipulation

- Scapy can easily parse certificates, and display their contents

---

In [ ]:
load_layer("tls")
cert_github = Cert(open("/content/drive/MyDrive/DataAnalysis/github.pem").read())  # assuming you d/l the certificate
cert_github

[X.509 Cert. Subject:/CN=github.com, Issuer:/C=GB/O=Sectigo Limited/CN=Sectigo Public Server Authentication CA DV E36]

- several useful methods help exploring them

---

In [ ]:
print(cert_github.isSelfSigned())  # check if it is self signed
print(cert_github.subject)  # display the subject
print(cert_github.remainingDays()) # compute the number of days until expiration

False
{'commonName': 'github.com'}
85.02425925925925


# TLS tricks

- sniff TLS traffic

---

In [ ]:
load_layer("tls")
#s = sniff(filter="port 443", count=10)  # sniff packets on port 443
s = rdpcap("/content/drive/MyDrive/DataAnalysis/tls.pcapng")
ch_list = [p for p in s if TLSClientHello in p]  # filter Client Hello messages
ch_list[0][TLSClientHello].show()  # display the first message

###[ TLS Handshake - Client Hello ]###
  msgtype   = client_hello
  msglen    = 774
  version   = TLS 1.2
  gmt_unix_time= Sun, 28 May 2102 20:12:20  (4178290340)
  random_bytes= db608a0658107674bc1b5515c6d65db4701f4db59e9ac8a4b0017930
  sidlen    = 32
  sid       = b')\xf0\x0e\x95\x0e \x86\xac>\xca\x97\xbdU\x8dn\x05Mt\xe0\x0c\x9b,;Ie\x19\xb3\x12\x1f\xa8\xe3\xf6'
  cipherslen= 40
  ciphers   = [TLS_AES_256_GCM_SHA384, TLS_AES_128_GCM_SHA256, TLS_ECDHE_ECDSA_WITH_AES_256_GCM_SHA384, TLS_ECDHE_ECDSA_WITH_AES_128_GCM_SHA256, TLS_ECDHE_RSA_WITH_AES_256_GCM_SHA384, TLS_ECDHE_RSA_WITH_AES_128_GCM_SHA256, TLS_ECDHE_ECDSA_WITH_AES_256_CBC_SHA384, TLS_ECDHE_ECDSA_WITH_AES_128_CBC_SHA256, TLS_ECDHE_RSA_WITH_AES_256_CBC_SHA384, TLS_ECDHE_RSA_WITH_AES_128_CBC_SHA256, TLS_ECDHE_ECDSA_WITH_AES_256_CBC_SHA, TLS_ECDHE_ECDSA_WITH_AES_128_CBC_SHA, TLS_ECDHE_RSA_WITH_AES_256_CBC_SHA, TLS_ECDHE_RSA_WITH_AES_128_CBC_SHA, TLS_RSA_WITH_AES_256_GCM_SHA384, TLS_RSA_WITH_AES_128_GCM_SHA256, TLS_RSA_WITH_AES_256